In [ ]:
import sys
sys.path.insert(0, '..')

import torch
import pandas as pd
from nucb_transformer.utils.io import load_checkpoint, best_checkpoint_path
from nucb_transformer.training.metrics import ACTIVITY_CLASSES, confusion_matrix_df
from scripts.predict import predict_sequences

In [ ]:
# Load the best checkpoint from the default directory
device = torch.device('cpu')
ckpt_path = best_checkpoint_path('../checkpoints/')
model, _ = load_checkpoint(ckpt_path, device='cpu')
print(f'Loaded: {ckpt_path}')

## Predict a single sequence

In [ ]:
sequence = 'PASTE_YOUR_SEQUENCE_HERE'

result = predict_sequences(model, [sequence], device)
result

## Predict a list of sequences

In [ ]:
sequences = [
    'SEQUENCE_1',
    'SEQUENCE_2',
]

results = predict_sequences(model, sequences, device)
results

## Load and explore test set predictions
Run `python scripts/evaluate.py --save_predictions results/test_predictions.csv` first.

In [ ]:
test_df = pd.read_csv('../results/test_predictions.csv')
test_df.head()

In [ ]:
# Accuracy by number of mutations
test_df.groupby('num_mutations')['correct'].mean().rename('accuracy')

In [ ]:
# High-activity variants the model missed
missed = test_df[
    test_df['activity_level'].isin(['activity > WT', 'activity > A73R']) &
    ~test_df['correct']
]
missed[['sequence', 'num_mutations', 'activity_level', 'predicted_class']]

In [ ]:
# Confusion matrix
import numpy as np
from nucb_transformer.training.metrics import ACTIVITY_CLASSES

label_to_idx = {c: i for i, c in enumerate(ACTIVITY_CLASSES)}
preds_np  = test_df['predicted_class'].map(label_to_idx).values
labels_np = test_df['activity_level'].map(label_to_idx).values

confusion_matrix_df(preds_np, labels_np)